# Langchain router
It uses RunnableBranch to decide what branch to run



In [ ]:
%pip install -qU langchain-ollama --quiet

Import libraries

In [1]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

Initializing the model

In [2]:
llm = ChatOllama(
    model="gemma4:e4b",
    temperature=0.1,
)

## Verbosity setting
The user will select a verbosity mode (short, normal, long) in the GUI and the variable 'verbosity' will be assigned and passed to trigger the relevant branch in the chain. The verbosity mode will be used to control the amount of information printed

In [3]:
# Uncomment one of the following for testing
#verbosity = "short"
#verbosity = "normal"
verbosity = "long"

## Prompt templates for all verbosities

In [4]:
short_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful and concise AI assistant. Respond concisely in 50 words or less."),
    ("user", "{input}")
    ])

In [5]:
normal_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Provide a structured paragraph of roughly 100 words."),
    ("user", "{input}")
    ])

In [6]:
long_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful and very verbose AI assistant. Respond with 3 or more paragraphs with lots of details and examples."),
    ("user", "{input}")
    ])

## Chains for all branches

In [7]:
short_chain = short_prompt | llm | StrOutputParser() | RunnableLambda(lambda x: { "answer": x , "verbosity": verbosity })
normal_chain = normal_prompt | llm | StrOutputParser() | RunnableLambda(lambda x: { "answer": x , "verbosity": verbosity })
long_chain = long_prompt | llm | StrOutputParser() | RunnableLambda(lambda x: { "answer": x , "verbosity": verbosity })

## Boolean functions to pick the right chain based on verbosity level
Normal will trigger the default branch, so we won't need a function 

In [8]:
def is_short(verbosity):
    return verbosity == "short"

def is_long(verbosity):
    return verbosity == "long"

print("Short = ",is_short(verbosity))
print("Long = ",is_long(verbosity))

Short =  False
Long =  True


## Main chains

In [9]:
router = RunnableBranch(
    (is_short, short_chain),
    (is_long, long_chain),
    normal_chain 
) 

Test the chain with "invoke" method

In [10]:
verbosity = "long"
response = router.invoke({"input": "How does photosynthesis work?", "verbosity": verbosity})


In [11]:
import textwrap
line_width = 120
print(textwrap.fill(response["answer"], line_width))
print("Verbosity used: ", response["verbosity"].upper())

Photosynthesis is the vital biochemical process by which plants, algae, and certain bacteria convert light energy into
chemical energy. This process primarily occurs within chloroplasts, which contain the pigment chlorophyll, enabling the
capture of sunlight. Plants take in carbon dioxide ($\text{CO}_2$) from the air and absorb water ($\text{H}_2\text{O}$)
through their roots. Using the captured solar energy, the plant converts these simple inorganic molecules into glucose
(a sugar that serves as food) and releases oxygen ($\text{O}_2$) as a crucial byproduct. Essentially, photosynthesis
fuels nearly all life on Earth, forming the base of most food chains.
Verbosity used:  LONG


In [12]:
verbosity = "normal"
response = router.invoke({"input": "How does photosynthesis work?", "verbosity": verbosity})

print(textwrap.fill(response["answer"], line_width))
print("Verbosity used: ", response["verbosity"].upper())

Photosynthesis is the vital biochemical process by which plants, algae, and some bacteria convert light energy into
chemical energy. Using chlorophyll, a pigment found in chloroplasts, these organisms capture sunlight, which powers the
reaction. They take in carbon dioxide ($\text{CO}_2$) from the air and water ($\text{H}_2\text{O}$) through their roots.
Through a complex series of steps, the plant converts these simple inputs into glucose (a sugar that serves as food) and
releases oxygen ($\text{O}_2$) as a crucial byproduct. This process sustains nearly all life on Earth, forming the base
of most food chains.
Verbosity used:  NORMAL


In [13]:
verbosity = "short"
response = router.invoke({"input": "How does photosynthesis work?", "verbosity": verbosity})

print(textwrap.fill(response["answer"], line_width))
print("Verbosity used: ", response["verbosity"].upper())

Photosynthesis is the vital biochemical process by which plants, algae, and some bacteria convert light energy into
chemical energy. Using chlorophyll, a pigment found in chloroplasts, these organisms capture sunlight, which powers the
reaction. They take in carbon dioxide ($\text{CO}_2$) from the air and water ($\text{H}_2\text{O}$) through the roots.
Through a complex series of chemical reactions, the plant converts these simple inputs into glucose (a sugar that serves
as food) and releases oxygen ($\text{O}_2$) as a crucial byproduct. This process sustains nearly all life on Earth,
forming the base of most food chains.
Verbosity used:  SHORT
